In [1]:
from IPython.display import display, HTML
display(HTML('''
<style>
.jp-Cell-outputWrapper .jp-Placeholder {
    display: none;
}
</style>
'''))

<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2024</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Типы - Часть 2</b><br/>
    <br/>
    <font>Никита Бондарцев</font><br/>
</center>

### Что мы сегодня разберем?

### Специальный тип Any

In [2]:
import typing as tp
print(tp.Any.__doc__.split("\n")[0])

Special type indicating an unconstrained type.


### Почему тип Any специальный?

In [3]:
import typing as tp

# Тип-класс
class A:
    pass

# Тип-класс
B = int

# Тип-объект
C = tp.Any

type(A), type(B), type(C)

(type, type, typing._AnyMeta)

In [4]:
isinstance(1, tp.Any)

TypeError: typing.Any cannot be used with isinstance()

### Специальный тип Union

In [5]:
import typing as tp
# нынче, с питона 3.10, можно писать просто через вертикальную черту A | B
print(tp.Union.__doc__.split("\n")[0])

Union type; Union[X, Y] means either X or Y.


Определение подтипа:
* ∀ A: A -> A
* A -> B  =>  A.values ⊇ B.values
* A -> B  =>  A.functions ⊆ B.functions

float | str => объединение всех значений, пересечение всех методов, поэтому

#### Выполняются ли такие свойства?

float | str -> float \
float | str -> str

In [6]:
%%typecheck

class A:
    def am(self) -> None:
        pass
    
    def run(self) -> None:
        pass

class B:
    def run(self) -> None:
        pass


a: A | B = A()
reveal_type(a)
a.am()
a.run()

<string>:16: note: Revealed type is "Union[__main__.A, __main__.B]"
<string>:17: error: Item "B" of "A | B" has no attribute "am"  [union-attr]
Found 1 error in 1 file (checked 1 source file)



In [7]:
float | str == str | float, tp.Union[str, float] == str | float

(True, True)

In [8]:
float == tp.Union[float]

True

### Union → Примеры

In [10]:
%%typecheck
# Использование надтипов в Union

def f(a: float | str) -> None:
    pass

f(2)
f(2.0)
f("hello")
f({})

<string>:10: error: Argument 1 to "f" has incompatible type "dict[Never, Never]"; expected "float | str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



In [11]:
%%typecheck
# Использование подтипов в Union

def f(a: int | str) -> None:
    pass

f(2)
f(2.0)
f("hello")
f({})

<string>:8: error: Argument 1 to "f" has incompatible type "float"; expected "int | str"  [arg-type]
<string>:10: error: Argument 1 to "f" has incompatible type "dict[Never, Never]"; expected "int | str"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)



In [12]:
%%typecheck
# Вывод типов с учетом isinstance

def f(a: int | float | str) -> None:
    reveal_type(a)
    if isinstance(a, int):
        reveal_type(a)
    else:
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
        else:
            reveal_type(a)


<string>:5: note: Revealed type is "Union[builtins.int, builtins.float, builtins.str]"
<string>:7: note: Revealed type is "builtins.int"
<string>:9: note: Revealed type is "Union[builtins.float, builtins.str]"
<string>:11: note: Revealed type is "builtins.str"
<string>:13: note: Revealed type is "builtins.float"
Success: no issues found in 1 source file



#### Some really weird things (will be discussed again later just before Protocols, but it would still be not clear, sorry)

- [virtual subclassing of ABC](https://docs.python.org/3.12/library/abc.html#abc.ABCMeta.register)
- the [numbers module](https://docs.python.org/3.12/library/numbers.html) and ABC hierarchy, [pep for numeric tower](https://peps.python.org/pep-3141/)
- source code that [registers float](https://github.com/python/cpython/blob/3.12/Lib/numbers.py#L289) as a virtual subclass of numbers.Real
- isinstance() would not work as expected!

In [18]:
class A: pass
class B(A): pass


issubclass(int, float), isinstance(2, float), isinstance(2., float), issubclass(B, A), isinstance(B(), A)


(False, False, True, True, True)

In [23]:
%%typecheck
# Вывод типов с учетом isinstance
# --warn-unreachable

def f(a: int | float | str) -> None:
    reveal_type(a)
    if isinstance(a, float):
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
    else:
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
        else:
            reveal_type(a)

<string>:6: note: Revealed type is "Union[builtins.int, builtins.float, builtins.str]"
<string>:8: note: Revealed type is "builtins.float"
<string>:9: error: Subclass of "float" and "str" cannot exist: would have incompatible method signatures  [unreachable]
<string>:10: error: Statement is unreachable  [unreachable]
<string>:12: note: Revealed type is "Union[builtins.int, builtins.str]"
<string>:14: note: Revealed type is "builtins.str"
<string>:16: note: Revealed type is "builtins.int"
Found 2 errors in 1 file (checked 1 source file)



### Специальный тип Optional

In [24]:
import typing as tp
print(tp.Optional.__doc__.split("\n")[0])

Optional[X] is equivalent to Union[X, None].


In [25]:
import typing as tp
from types import NoneType  # python 3.10+

tp.Optional[float] == float | None, tp.Optional[float] == float | NoneType, type(None), type(None) is NoneType

(True, True, NoneType, True)

### Специальный тип Optional, примеры

In [26]:
%%typecheck

def f(a: float | None) -> None:
    pass

f(1)
f(1.0)
f("1")
f(None)

<string>:8: error: Argument 1 to "f" has incompatible type "str"; expected "float | None"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



In [27]:
%%typecheck
# reveal в Optional делается аналогично Union

def f(a: float | None) -> None:
    reveal_type(a)
    if a is None:
        reveal_type(a)
    else:
        reveal_type(a)

<string>:5: note: Revealed type is "Union[builtins.float, None]"
<string>:7: note: Revealed type is "None"
<string>:9: note: Revealed type is "builtins.float"
Success: no issues found in 1 source file



### Что такое генерики?

Генерик - параметризированный тип, [ссылка на доку](https://mypy.readthedocs.io/en/stable/generics.html#defining-generic-classes)

Например, \
List[int] - тип list с параметром int \
Dict[str, int] - dict c параметром str у ключа и int у значения \
Tuple[int, ...] - tuple с параметрами int

Note: с питона 3.9 можно использовать стандартные контейнеры в качестве генериков, например, list[int], dict[str, int], tuple[int, float, str]

### Ковариантность/контрвариантность

### Примеры вариантности

### tuple (typing.Tuple, deprecated)

In [29]:
from typing import Tuple
print(Tuple.__doc__)

Deprecated alias to builtins.tuple.

    Tuple[X, Y] is the cross-product type of X and Y.

    Example: Tuple[T1, T2] is a tuple of two elements corresponding
    to type variables T1 and T2.  Tuple[int, float, str] is a tuple
    of an int, a float and a string.

    To specify a variable-length tuple of homogeneous type, use Tuple[T, ...].
    


In [30]:
%%typecheck
import typing as tp

a: tp.Tuple[int, str] = (1, "hello")
b: tuple[int, int, int] = (1, 2, 3)
c: tuple[int, ...] = (1, 2, 3, 4, 5)
d: tuple[int, float, str] = (1, 2.)

<string>:7: error: Incompatible types in assignment (expression has type "tuple[int, float]", variable has type "tuple[int, float, str]")  [assignment]
Found 1 error in 1 file (checked 1 source file)



### tuple, пример

In [32]:
%%typecheck
# Heterogeneus tuple 

def f(a: tuple[int, float]) -> None:
    a[0] << 10
    a[1] << 10
    a[2] + 1

f((1, 1.4))
f((1, 1.4, 1))
f((1, 1))
f((1.4, 1))
f((1, "1"))

<string>:6: error: Unsupported operand types for << ("float" and "int")  [operator]
<string>:7: error: Tuple index out of range  [misc]
<string>:10: error: Argument 1 to "f" has incompatible type "tuple[int, float, int]"; expected "tuple[int, float]"  [arg-type]
<string>:12: error: Argument 1 to "f" has incompatible type "tuple[float, int]"; expected "tuple[int, float]"  [arg-type]
<string>:13: error: Argument 1 to "f" has incompatible type "tuple[int, str]"; expected "tuple[int, float]"  [arg-type]
Found 5 errors in 1 file (checked 1 source file)



In [33]:
%%typecheck
# Homogeneus tuple 

def f(a: tuple[float, ...]) -> None:
    a[0] + 1
    a[1] << 10
    a[100500] = 1.54

f((1, 2, 3, 4, 5))
f((1.1, 2.1, 3.1, 4.1, 5.1))
f(("3", "2", "1"))
f(tuple())

<string>:6: error: Unsupported operand types for << ("float" and "int")  [operator]
<string>:7: error: Unsupported target for indexed assignment ("tuple[float, ...]")  [index]
<string>:11: error: Argument 1 to "f" has incompatible type "tuple[str, str, str]"; expected "tuple[float, ...]"  [arg-type]
Found 3 errors in 1 file (checked 1 source file)



### Какая вариантность у tuple[T]?

### list (typing.List, not deprecated, hmm)

In [34]:
import typing as tp
print(tp.List.__doc__)

A generic version of list.


### list, примеры

In [35]:
%%typecheck

a = [1, "hello", 5.1]
reveal_type(a)

a.append(1)
a.append(1.0)
a.append("hi")
a.append([])

<string>:4: note: Revealed type is "builtins.list[builtins.object]"
Success: no issues found in 1 source file



In [36]:
%%typecheck

a = []
reveal_type(a)

<string>:3: error: Need type annotation for "a" (hint: "a: list[<type>] = ...")  [var-annotated]
<string>:4: note: Revealed type is "builtins.list[Any]"
Found 1 error in 1 file (checked 1 source file)



In [37]:
%%typecheck

a: list[float] = []

a.append(1)
reveal_type(a)

<string>:6: note: Revealed type is "builtins.list[builtins.float]"
Success: no issues found in 1 source file



In [38]:
%%typecheck

a: list[int] = []

a.append(1.1)
reveal_type(a)

<string>:5: error: Argument 1 to "append" of "list" has incompatible type "float"; expected "int"  [arg-type]
<string>:6: note: Revealed type is "builtins.list[builtins.int]"
Found 1 error in 1 file (checked 1 source file)



In [39]:
%%typecheck
# Повышение типа параметра

def foo(a: list[int]) -> None:
    pass

my_list = [1.1, 3.1, 5.1]
reveal_type(my_list)

foo(my_list)

<string>:8: note: Revealed type is "builtins.list[builtins.float]"
<string>:10: error: Argument 1 to "foo" has incompatible type "list[float]"; expected "list[int]"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



In [40]:
%%typecheck
# Понижение типа параметра

def foo(a: list[float]) -> None:
    pass

my_list = [1, 2, 3]
reveal_type(my_list)

foo(my_list)

<string>:8: note: Revealed type is "builtins.list[builtins.int]"
<string>:10: error: Argument 1 to "foo" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
<string>:10: note: "List" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:10: note: Consider using "Sequence" instead, which is covariant
Found 1 error in 1 file (checked 1 source file)



### Какая вариантность у list[T]?

### typing.Sequence/typing.Mapping

### Sequence, пример

In [41]:
%%typecheck

def foo(a: list[float]) -> float:
    return a[0]

my_list = [1, 3, 5]

foo(my_list)

<string>:8: error: Argument 1 to "foo" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
<string>:8: note: "List" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:8: note: Consider using "Sequence" instead, which is covariant
Found 1 error in 1 file (checked 1 source file)



In [42]:
%%typecheck

import collections.abc as abc

def foo(a: abc.Sequence[float]) -> float:
    return a[0]

my_list = [1, 3, 5]

foo(my_list)

Success: no issues found in 1 source file



### abc.Mapping, пример

In [43]:
%%typecheck

def foo(a: dict[str, float]) -> float:
    return a["key"]

my_dict = {"hey": 1}

foo(my_dict)

<string>:8: error: Argument 1 to "foo" has incompatible type "dict[str, int]"; expected "dict[str, float]"  [arg-type]
<string>:8: note: "Dict" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:8: note: Consider using "Mapping" instead, which is covariant in the value type
Found 1 error in 1 file (checked 1 source file)



In [44]:
%%typecheck

import collections.abc as abc

def foo(a: abc.Mapping[str, float]) -> float:
    return a["key"]

my_dict = {"hey": 1}

foo(my_dict)

Success: no issues found in 1 source file



### Иерархия типов генериков
<img src="images/hierarchy2.jpg">

### Примеры на иерархию abc.Sequence/abc.Mapping 

In [45]:
%%typecheck

import collections.abc as abc


def foo1(a: abc.Sequence[float]) -> None:
    pass

def foo2(a: abc.MutableSequence[float]) -> None:
    pass

def foo3(a: list[float]) -> None:
    pass

def foo4(a: list) -> None:
    pass


a = [1]
foo1(a)
foo2(a)
foo3(a)
foo4(a)


<string>:15: error: Missing type parameters for generic type "list"  [type-arg]
<string>:21: error: Argument 1 to "foo2" has incompatible type "list[int]"; expected "MutableSequence[float]"  [arg-type]
<string>:22: error: Argument 1 to "foo3" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
<string>:22: note: "List" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:22: note: Consider using "Sequence" instead, which is covariant
Found 3 errors in 1 file (checked 1 source file)



In [46]:
%%typecheck

import collections.abc as abc


def foo1(a: abc.Mapping[str, float]) -> None:
    pass

def foo2(a: abc.MutableMapping[str, float]) -> None:
    pass

def foo3(a: dict[str, float]) -> None:
    pass

def foo4(a: dict) -> None:
    pass


a = {'a': 1}
foo1(a)
foo2(a)
foo3(a)
foo4(a)

<string>:15: error: Missing type parameters for generic type "dict"  [type-arg]
<string>:21: error: Argument 1 to "foo2" has incompatible type "dict[str, int]"; expected "MutableMapping[str, float]"  [arg-type]
<string>:22: error: Argument 1 to "foo3" has incompatible type "dict[str, int]"; expected "dict[str, float]"  [arg-type]
<string>:22: note: "Dict" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:22: note: Consider using "Mapping" instead, which is covariant in the value type
Found 3 errors in 1 file (checked 1 source file)



### Generic functions, TypeVar

In [48]:
import typing
print(tp.TypeVar.__doc__)

Type variable.

The preferred way to construct a type variable is via the dedicated
syntax for generic functions, classes, and type aliases::

    class Sequence[T]:  # T is a TypeVar
        ...

This syntax can also be used to create bound and constrained type
variables::

    # S is a TypeVar bound to str
    class StrSequence[S: str]:
        ...

    # A is a TypeVar constrained to str or bytes
    class StrOrBytesSequence[A: (str, bytes)]:
        ...

However, if desired, reusable type variables can also be constructed
manually, like so::

   T = TypeVar('T')  # Can be anything
   S = TypeVar('S', bound=str)  # Can be any subtype of str
   A = TypeVar('A', str, bytes)  # Must be exactly str or bytes

Type variables exist primarily for the benefit of static type
checkers.  They serve as the parameters for generic types as well
as for generic function and type alias definitions.

The variance of type variables is inferred by type checkers when they
are created through the type par

#### TypeVar, мотивация

In [49]:
%%typecheck

def foo(a: int, b: int) -> int:
    reveal_type(a)
    return a + b

a = foo(1, 2)
reveal_type(a)

<string>:4: note: Revealed type is "builtins.int"
<string>:8: note: Revealed type is "builtins.int"
Success: no issues found in 1 source file



In [50]:
%%typecheck

def foo(a: int, b: int) -> int:
    return a + b

def foo2(a: str, b: str) -> str:
    return a + b

a = foo(1, 2)
reveal_type(a)

b = foo2("1", "2")
reveal_type(b)

c = foo("1", 2)
reveal_type(c)

<string>:10: note: Revealed type is "builtins.int"
<string>:13: note: Revealed type is "builtins.str"
<string>:15: error: Argument 1 to "foo" has incompatible type "str"; expected "int"  [arg-type]
<string>:16: note: Revealed type is "builtins.int"
Found 1 error in 1 file (checked 1 source file)



In [51]:
%%typecheck

def foo(a: int | str, b: int | str) -> int | str:
    reveal_type(a)
    return a + b

a = foo(1, 2)
reveal_type(a)

b = foo("1", "2")
reveal_type(b)

c = foo("1", 2)
reveal_type(c)


<string>:4: note: Revealed type is "Union[builtins.int, builtins.str]"
<string>:5: error: Unsupported operand types for + ("int" and "str")  [operator]
<string>:5: error: Unsupported operand types for + ("str" and "int")  [operator]
<string>:5: note: Both left and right operands are unions
<string>:8: note: Revealed type is "Union[builtins.int, builtins.str]"
<string>:11: note: Revealed type is "Union[builtins.int, builtins.str]"
<string>:14: note: Revealed type is "Union[builtins.int, builtins.str]"
Found 2 errors in 1 file (checked 1 source file)



### TypeVar, примеры

In [52]:
%%typecheck

import typing as tp

T = tp.TypeVar('T', int, str)

def foo(a: T, b: T) -> T:
    reveal_type(a)
    return a + b

a = foo(1, 2)
reveal_type(a)

b = foo("1", "2")
reveal_type(b)

c = foo("1", 2)
reveal_type(c)

<string>:8: note: Revealed type is "builtins.int"
<string>:8: note: Revealed type is "builtins.str"
<string>:12: note: Revealed type is "builtins.int"
<string>:15: note: Revealed type is "builtins.str"
<string>:17: error: Value of type variable "T" of "foo" cannot be "object"  [type-var]
<string>:18: note: Revealed type is "builtins.object"
Found 1 error in 1 file (checked 1 source file)



In [55]:
%%typecheck
# новый синтаксис с 3.12, enable_incomplete_feature = "NewGenericSyntax"

def foo[T: (int, str)](a: T, b: T) -> T:
    reveal_type(a)
    return a + b

a = foo(1, 2)
reveal_type(a)

b = foo("1", "2")
reveal_type(b)

c = foo("1", 2)
reveal_type(c)

<string>:5: note: Revealed type is "builtins.int"
<string>:5: note: Revealed type is "builtins.str"
<string>:9: note: Revealed type is "builtins.int"
<string>:12: note: Revealed type is "builtins.str"
<string>:14: error: Value of type variable "T" of "foo" cannot be "object"  [type-var]
<string>:15: note: Revealed type is "builtins.object"
Found 1 error in 1 file (checked 1 source file)



In [56]:
%%typecheck

def foo[T](a: list[T], n: int) -> T:
    return a[n]

a = foo([1, 2], 0)
reveal_type(a)

<string>:7: note: Revealed type is "builtins.int"
Success: no issues found in 1 source file



### Overload - не то, что вы подумали
[дока](https://peps.python.org/pep-0484/#function-method-overloading)

In [57]:
%%typecheck

lst = [1, 2, 3]
reveal_type(lst[0])
reveal_type(lst[0:1])

<string>:4: note: Revealed type is "builtins.int"
<string>:5: note: Revealed type is "builtins.list[builtins.int]"
Success: no issues found in 1 source file



In [59]:
%%typecheck

import typing as tp

class MyIntList:
    def __init__(self, lst: list[int]) -> None:
        self.lst = lst

    @tp.overload
    def __getitem__(self, idx: slice) -> list[int]:
        pass

    @tp.overload
    def __getitem__(self, idx: int) -> int:
        pass
    
    def __getitem__(self, idx: slice | int) -> list[int] | int:
        return self.lst[idx]

my_lst = MyIntList([1, 2, 3])
reveal_type(my_lst[0])
reveal_type(my_lst[0:1])

<string>:21: note: Revealed type is "builtins.int"
<string>:22: note: Revealed type is "builtins.list[builtins.int]"
Success: no issues found in 1 source file



### Функциональный тип Callable

In [60]:
%%typecheck

import collections.abc as abc

def g(a: int, b: float) -> float:
    return 1.1

a: abc.Callable[[int, float], float] = g

Success: no issues found in 1 source file



### Callable, аннотация типов функции совпадает с тайпингом

In [61]:
%%typecheck
# Привычная функция

import collections.abc as abc

def f(a: abc.Callable[[int, float], float]) -> None:
    pass

def g(a: int, b: float) -> float:
    return 1.1

f(g)

Success: no issues found in 1 source file



In [63]:
%%typecheck
# Функция без аргументов

import collections.abc as abc

def f(a: abc.Callable[[], float]) -> None:
    pass

def g() -> float:
    return 1.1

f(g)

Success: no issues found in 1 source file



### Callable, часть типов отсутствует

In [64]:
%%typecheck
# Любые аргументы

import collections.abc as abc

def f(a: abc.Callable[..., float]) -> None:
    pass

def g(a: int, b: str, c: int) -> float:
    return 1.1

f(g)

Success: no issues found in 1 source file



In [65]:
%%typecheck
# И лямбда тоже

import typing as tp

def f(a: tp.Callable[..., float]) -> None:
    pass

f(lambda a, b, c: 1.1)

Success: no issues found in 1 source file



### Callable, передаем функции с другими типами аргументов

In [66]:
%%typecheck
# Повышаем тип аргумента

import collections.abc as abc

def f(a: abc.Callable[[int], float]) -> None:
    pass

def g(a: float) -> float:
    return a

f(g)

Success: no issues found in 1 source file



In [67]:
%%typecheck
# Понижаем тип аргумента

import collections.abc as abc

def f(a: abc.Callable[[float], float]) -> None:
    pass

def g(a: int) -> float:
    return a

f(g)

<string>:12: error: Argument 1 to "f" has incompatible type "Callable[[int], float]"; expected "Callable[[float], float]"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



### Callable, передаем функции с другим типом возвращаемого значения

In [68]:
%%typecheck
# Повышаем тип возвращаемого значения

import collections.abc as abc

def f(a: abc.Callable[[int], int]) -> None:
    pass

def g(a: int) -> float:
    return a

f(g)

<string>:12: error: Argument 1 to "f" has incompatible type "Callable[[int], float]"; expected "Callable[[int], int]"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



In [69]:
%%typecheck
# Понижаем тип возвращаемого значения

import collections.abc as abc

def f(a: abc.Callable[[int], float]) -> None:
    pass

def g(a: int) -> int:
    return a

f(g)

Success: no issues found in 1 source file



### Какая вариантность Callable по аргументам? По возвращаемому значению?

### Создание собственных Generic классов

In [71]:
import typing as tp
print(tp.Generic.__doc__)

Abstract base class for generic types.

On Python 3.12 and newer, generic classes implicitly inherit from
Generic when they declare a parameter list after the class's name::

    class Mapping[KT, VT]:
        def __getitem__(self, key: KT) -> VT:
            ...
        # Etc.

On older versions of Python, however, generic classes have to
explicitly inherit from Generic.

After a class has been declared to be generic, it can then be used as
follows::

    def lookup_name[KT, VT](mapping: Mapping[KT, VT], key: KT, default: VT) -> VT:
        try:
            return mapping[key]
        except KeyError:
            return default



### Generic, не так прост как кажется

In [72]:
%%typecheck
import typing as tp

# Используем тайпвар, как будто пишем дженерик функцию - не работает

T = tp.TypeVar("T", str, int)

class A:
    def __init__(self, a: T) -> None:
        self._a = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a + self._a

    
a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

<string>:10: error: Need type annotation for "_a"  [var-annotated]
<string>:11: note: Revealed type is "builtins.str"
<string>:11: note: Revealed type is "Any"
<string>:13: error: A function returning TypeVar should receive at least one argument containing the same TypeVar  [type-var]
<string>:14: note: Revealed type is "Any"
<string>:19: note: Revealed type is "__main__.A"
<string>:21: note: Revealed type is "builtins.str"
Found 2 errors in 1 file (checked 1 source file)



In [73]:
%%typecheck
import typing as tp

# корректно - наследоваться от tp.Generic[T]

T = tp.TypeVar("T", int, str)

class A(tp.Generic[T]):    
    def __init__(self, a: T) -> None:
        self._a: T = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a


a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

c = A("hello")
reveal_type(c)
d = c.am()
reveal_type(d)

<string>:9: note: Revealed type is "builtins.int"
<string>:9: note: Revealed type is "builtins.str"
<string>:12: note: Revealed type is "builtins.int"
<string>:12: note: Revealed type is "builtins.str"
<string>:17: note: Revealed type is "__main__.A[builtins.int]"
<string>:19: note: Revealed type is "builtins.int"
<string>:22: note: Revealed type is "__main__.A[builtins.str]"
<string>:24: note: Revealed type is "builtins.str"
Success: no issues found in 1 source file



In [74]:
%%typecheck

# новый синтаксис с 3.12

class A[T]:
    def __init__(self, a: T) -> None:
        self._a: T = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a


a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

c = A("hello")
reveal_type(c)
d = c.am()
reveal_type(d)

<string>:8: note: Revealed type is "T`1"
<string>:11: note: Revealed type is "T`1"
<string>:16: note: Revealed type is "__main__.A[builtins.int]"
<string>:18: note: Revealed type is "builtins.int"
<string>:21: note: Revealed type is "__main__.A[builtins.str]"
<string>:23: note: Revealed type is "builtins.str"
Success: no issues found in 1 source file



In [75]:
%%typecheck

import typing as tp

T = tp.TypeVar("T")

class A(tp.Generic[T]):
    def __init__(self) -> None:
        self._a: list[T] = []
        
    def add(self, a: T) -> None:
        self._a.append(a)


a: A[int] = A()
a.add(1)
reveal_type(a)

b: A[float] = A()
b.add("sss")

<string>:17: note: Revealed type is "__main__.A[builtins.int]"
<string>:20: error: Argument 1 to "add" of "A" has incompatible type "str"; expected "float"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



###  Переменные классов, Type

In [76]:
# https://www.python.org/dev/peps/pep-0484/#the-type-of-class-objects

import typing as tp
print(tp.Type.__doc__)

Deprecated alias to builtins.type.

    builtins.type or typing.Type can be used to annotate class objects.
    For example, suppose we have the following classes::

        class User: ...  # Abstract base for User classes
        class BasicUser(User): ...
        class ProUser(User): ...
        class TeamUser(User): ...

    And a function that takes a class argument that's a subclass of
    User and returns an instance of the corresponding class::

        def new_user[U](user_class: Type[U]) -> U:
            user = user_class()
            # (Here we could write the user object to a database)
            return user

        joe = new_user(BasicUser)

    At this point the type checker knows that joe has type BasicUser.
    


In [77]:
%%typecheck

import typing as tp

a: tp.Type[int] = int

class B:
    pass

b: tp.Type[B] = B

Success: no issues found in 1 source file



###  Type, пример

In [78]:
%%typecheck
import typing as tp

class A:
    pass

class B(A):
    pass

def foo(a: tp.Type[A]) -> None:
    pass


foo(A())
foo(A)
foo(B())
foo(B)

<string>:14: error: Argument 1 to "foo" has incompatible type "A"; expected "type[A]"  [arg-type]
<string>:16: error: Argument 1 to "foo" has incompatible type "B"; expected "type[A]"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)



In [79]:
%%typecheck
import typing as tp

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: tp.Type[A]) -> A:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

<string>:16: note: Revealed type is "__main__.A"
<string>:17: note: Revealed type is "__main__.A"
Success: no issues found in 1 source file



In [81]:
%%typecheck
import typing as tp

# from python 3.11

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: tp.Type[tp.Self]) -> tp.Self:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

<string>:16: note: Revealed type is "__main__.A"
<string>:17: note: Revealed type is "__main__.B"
Success: no issues found in 1 source file



### Ограничение TypeVar без необходимости перечислять все варианты, ограничение сверху

In [84]:
%%typecheck
import typing as tp

T = tp.TypeVar('T', bound="A")

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: tp.Type[T]) -> T:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

<string>:18: note: Revealed type is "__main__.A"
<string>:19: note: Revealed type is "__main__.B"
Success: no issues found in 1 source file



In [90]:
%%typecheck
import typing as tp

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build[T: "A"](cls: tp.Type[T]) -> T:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

<string>:16: note: Revealed type is "__main__.A"
<string>:17: note: Revealed type is "__main__.B"
Success: no issues found in 1 source file



###  Какая вариантность у Type?

### Типы → Nominal subtyping, direct inheritance

### Типы → Structural subtyping

### Structural subtyping, примеры

In [110]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )

        
validate_size([10, 11], 1)  # OK
validate_size(1010, 1)  # fail

<string>:6: error: Function is missing a type annotation for one or more arguments  [no-untyped-def]
Found 1 error in 1 file (checked 1 source file)



In [111]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )

validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size(1010, 1)  # fail


<string>:6: error: Function is missing a type annotation for one or more arguments  [no-untyped-def]
Found 1 error in 1 file (checked 1 source file)



In [112]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )


validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size({1, 3, 10}, 1)  # OK
validate_size(1010, 1)  # fail


<string>:6: error: Function is missing a type annotation for one or more arguments  [no-untyped-def]
Found 1 error in 1 file (checked 1 source file)



In [113]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

class A:
    def __len__(self):
        return 1

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )
        

validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size({1, 3, 10}, 1)  # OK
validate_size(A(), 1)  # OK
validate_size(1010, 1)  # fail


<string>:7: error: Function is missing a type annotation  [no-untyped-def]
<string>:10: error: Function is missing a type annotation for one or more arguments  [no-untyped-def]
Found 2 errors in 1 file (checked 1 source file)



### Собственный Structural Subtping - Protocol

In [116]:
%%typecheck

import typing as tp


class Closeable(tp.Protocol):
    def close(self) -> None:
        pass
    

class A:
    def close(self) -> None:
        print("Close enough")

class B:
    pass


c = open("my_file")


def foo(a: Closeable) -> None:
    reveal_type(a)
    a.close()


foo(A())
foo(B())
foo(c)

<string>:23: note: Revealed type is "__main__.Closeable"
<string>:28: error: Argument 1 to "foo" has incompatible type "B"; expected "Closeable"  [arg-type]
Found 1 error in 1 file (checked 1 source file)



### Protocol and isinstance, runtime_checkable

In [117]:
import typing as tp

# Note that instance checks are not 100% reliable statically, this is why this behavior is opt-in, see section on rejected ideas for examples.
# @see https://peps.python.org/pep-0544/#runtime-checkable-decorator-and-narrowing-types-by-isinstance

@tp.runtime_checkable
class Closeable(tp.Protocol):
    def close(self) -> None:
        pass


class A:
    def close(self):
        print("Close enough")

isinstance(A(), Closeable)

True

### Пример готового Generic + Protocol, просто чтобы вас запутать

In [118]:
%%typecheck

import typing as tp
import collections.abc as abc

def foo(s: abc.Iterable[int]) -> None:
    pass

class A:
    pass

class B[T]:
    def __iter__(self) -> abc.Iterator[T]:
        return iter([])
    


foo(A())

b: B[int] = B()
foo(b)

c: B[str] = B()
foo(c)

foo([])

<string>:18: error: Argument 1 to "foo" has incompatible type "A"; expected "Iterable[int]"  [arg-type]
<string>:24: error: Argument 1 to "foo" has incompatible type "B[str]"; expected "Iterable[int]"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)



### Еще разок про числовые типы

В целом, в питончике творится черная магия с числовыми типами. Есть специальная [иерархия абстрактных типов](https://peps.python.org/pep-3141/) для чисел. А обычные числовые типы являются не прямыми наследниками друг драга, а скорее [структурными](https://mypy.readthedocs.io/en/latest/duck_type_compatibility.html).

In [119]:
from numbers import Real, Integral

isinstance(1, Real), isinstance(1, Integral), isinstance(1.1, Integral)

(True, True, False)

In [120]:
issubclass(float, Real), issubclass(int, float), issubclass(float, Integral)

(True, False, False)

Если кому-то интересно что за чертовщина происходит в ячейке ниже, велкам [в доку](https://peps.python.org/pep-0544/#existing-approaches-to-structural-subtyping). А вообще, с хеллоуином вас, товарищи!

In [121]:
%%typecheck

# НЕ РАБОТАЕТ для статической проверке типов, только в рантайме. float -> int это чит, которые засунут глубоко в язык

from abc import ABC, abstractmethod

class MyAbstract(ABC):
    @abstractmethod
    def haha(self) -> str:
        pass

class A:
    pass

MyAbstract.register(A)  # does not work for static type checking! Only for runtime

isinstance(A(), MyAbstract)  # True in runtime
issubclass(A, MyAbstract)  # True in runtime


def func(a: MyAbstract) -> None:
    pass

func(A())


<string>:24: error: Argument 1 to "func" has incompatible type "A"; expected "MyAbstract"  [arg-type]
Found 1 error in 1 file (checked 1 source file)

